## Conversion of ChEBI IDs

ChEBI entities are sometimes listed with their primary ID, and sometimes with their secondary ID. This notebook aims to create a relationship between all primary and secondary IDs in order to ensure correct mapping throughout the master. Finally, a function for the conversion from secondary to primary (s2p) ID is provided.

In [7]:
import pandas as pd
from rdkit import Chem
import numpy as np

In [2]:
def extract_id_mappings(sdf_file):
    supplier = Chem.SDMolSupplier(sdf_file)
    
    data = []
    for mol in supplier:
        if mol is None:
            continue
        
        primary_id = mol.GetProp("ChEBI ID")
        
        if mol.HasProp("Secondary ChEBI ID"):
            secondary_ids = mol.GetProp("Secondary ChEBI ID").split("\n")
        else:
            secondary_ids = []
        
        for sec_id in secondary_ids:
            data.append({"Primary_ID": primary_id, "Secondary_ID": sec_id})
    
    df = pd.DataFrame(data)
    return df

A lot of warnings and errors occur, but the whenever there are primary and secondary ID relations, they are extracted.

In [8]:
sdf_file = "../files/ChEBI_complete_3star.sdf"
mappings_df = extract_id_mappings(sdf_file)
mappings_df.to_csv("s2p.tsv", sep="\t", index=False)

[16:49:53] WARNING: not removing hydrogen atom without neighbors
[16:49:54] WARNING: not removing hydrogen atom with dummy atom neighbors
[16:49:54] WARNING: not removing hydrogen atom with dummy atom neighbors
[16:49:54] WARNING: not removing hydrogen atom with dummy atom neighbors
[16:49:54] Can't kekulize mol.  Unkekulized atoms: 1 3 5 7 8
[16:49:54] ERROR: Could not sanitize molecule ending on line 3032149
[16:49:54] ERROR: Can't kekulize mol.  Unkekulized atoms: 1 3 5 7 8
[16:49:55] Warning: molecule is tagged as 3D, but all Z coords are zero and 2D stereo markers have been found, marking the mol as 2D.
[16:49:55] 

****
Post-condition Violation
Element 'hv' not found
Violation occurred on line 93 in file C:\rdkit\build\temp.win-amd64-cpython-311\Release\rdkit\Code\GraphMol\PeriodicTable.h
Failed Expression: anum > -1
****

[16:49:55] ERROR: Element 'hv' not found
[16:49:55] ERROR: moving to the beginning of the next molecule
[16:49:55] Explicit valence for atom # 1 N, 4, is great

Following is a function that can be copied and used throughout the project for s2p conversion. This function returns exactly what was given, if it can't be found in the s2p.tsv file. This is because s2p.tsv only contain the ChEBI IDs where there are secondary IDs. And far from all entities have secondary IDs. Else, it always returns the primary ID of the submitted ID.

In [27]:
df = pd.read_csv("s2p.tsv", sep="\t")
secondary_to_primary = dict(zip(df["Secondary_ID"], df["Primary_ID"]))
def s2p(chebi_id):
    return secondary_to_primary.get(chebi_id, chebi_id)

Below is a funciton to extract ChEBI Name from the primary ID.

In [9]:
def extract_id_name_mapping(sdf_file):
    supplier = Chem.SDMolSupplier(sdf_file)
    
    data = []
    for mol in supplier:
        if mol is None:
            continue
        
        primary_id = mol.GetProp("ChEBI ID")
        
        if mol.HasProp("ChEBI Name"):
            chebi_name = mol.GetProp("ChEBI Name")
        else:
            chebi_name = np.nan
        

        data.append({"Primary_ID": primary_id, "ChEBI Name": chebi_name})
    
    df = pd.DataFrame(data)
    return df

In [10]:
sdf_file = "../files/ChEBI_complete_3star.sdf"
mappings_df = extract_id_name_mapping(sdf_file)
mappings_df.to_csv("p2n.tsv", sep="\t", index=False)

[15:02:08] WARNING: not removing hydrogen atom without neighbors
[15:02:10] WARNING: not removing hydrogen atom with dummy atom neighbors
[15:02:10] WARNING: not removing hydrogen atom with dummy atom neighbors
[15:02:11] WARNING: not removing hydrogen atom with dummy atom neighbors
[15:02:11] Can't kekulize mol.  Unkekulized atoms: 1 3 5 7 8
[15:02:11] ERROR: Could not sanitize molecule ending on line 3032149
[15:02:11] ERROR: Can't kekulize mol.  Unkekulized atoms: 1 3 5 7 8
[15:02:12] Warning: molecule is tagged as 3D, but all Z coords are zero and 2D stereo markers have been found, marking the mol as 2D.
[15:02:13] 

****
Post-condition Violation
Element 'hv' not found
Violation occurred on line 93 in file C:\rdkit\build\temp.win-amd64-cpython-311\Release\rdkit\Code\GraphMol\PeriodicTable.h
Failed Expression: anum > -1
****

[15:02:13] ERROR: Element 'hv' not found
[15:02:13] ERROR: moving to the beginning of the next molecule
[15:02:13] Explicit valence for atom # 1 N, 4, is great

In [ ]:
df = pd.read_csv("p2n.tsv", sep="\t")
primary_to_name = dict(zip(df["ChEBI Name"], df["Primary_ID"]))
def p2n(chebi_id):
    return primary_to_name(chebi_id, chebi_id)